# RG3 gas: relational features and nested defect similarity
Predefined hypotheses; outer and inner observation-block separation for learned scores.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import RobustScaler
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import Ridge
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import average_precision_score,confusion_matrix,precision_score,recall_score,f1_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
original=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
data=original.loc[original['product'].eq('RG3') & (original.PassOrFail.eq(0)|original.Reason.str.contains('가스',regex=False))].reset_index(drop=True)
y=data.Reason.str.contains('가스',regex=False).astype(int)
assert len(data)==583 and y.sum()==17
groups=data.comparison_block
# Predefined sensor groups, not selected by held-out gas performance.
sensor_groups={
'pressure':['Average_Back_Pressure','Max_Injection_Pressure','Max_Switch_Over_Pressure'],
'thermal':['Barrel_Temperature_2','Barrel_Temperature_5','Hopper_Temperature','Mold_Temperature_4'],
'filling':['Max_Injection_Speed','Filling_Time','Injection_Time','Cushion_Position'],
'plasticizing':['Plasticizing_Time','Average_Screw_RPM','Plasticizing_Position']}
relations={
'pressure_residual':('Max_Injection_Pressure',['Max_Injection_Speed','Filling_Time']),
'plasticizing_residual':('Plasticizing_Time',['Average_Screw_RPM','Average_Back_Pressure']),
'cushion_residual':('Cushion_Position',['Plasticizing_Position','Max_Injection_Pressure']),
'mold_residual':('Mold_Temperature_4',['Mold_Temperature_3','Barrel_Temperature_5'])}
static=pd.DataFrame(index=data.index)
static['barrel_front_shape']=data.Barrel_Temperature_1-data.Barrel_Temperature_3
static['barrel45_shape']=data.Barrel_Temperature_4-data.Barrel_Temperature_5
static['barrel56_shape']=data.Barrel_Temperature_5-data.Barrel_Temperature_6
static['mold_side_difference']=data.Mold_Temperature_4-data.Mold_Temperature_3
static['injection_minus_filling']=data.Injection_Time-data.Filling_Time
static['cycle_minus_plasticizing']=data.Cycle_Time-data.Plasticizing_Time
static['plasticizing_cycle_fraction']=data.Plasticizing_Time/data.Cycle_Time.where(data.Cycle_Time.ne(0))
static['filling_injection_fraction']=data.Filling_Time/data.Injection_Time.where(data.Injection_Time.ne(0))
shape=list(static.columns[:4]);phase=list(static.columns[4:])
distcols=[g+'_normal_distance' for g in sensor_groups]
nearcols=[g+'_gas_similarity' for g in sensor_groups]
rescols=list(relations)
oldcols=['Barrel_Temperature_2__past10_std','Hopper_Temperature__past10_std','Cushion_Position__past10_std','Barrel_Temperature_5__past10_std']
for c in oldcols:static[c]=data[c]
rawcols=['Average_Back_Pressure','Max_Injection_Pressure','Cushion_Position','Barrel_Temperature_5']
for c in rawcols:static[c]=data[c]
configs={
'thermal_shape4':shape,
'time_phase4':phase,
'normal_relation_residual4':rescols,
'normal_multivariate_distance4':distcols,
'gas_similarity4':nearcols,
'pressure_gas_thermal_shape3':['pressure_gas_similarity','barrel45_shape','barrel56_shape'],
'filling_plasticizing_similarity2':['filling_gas_similarity','plasticizing_gas_similarity'],
'pressure_filling_similarity_residual4':['pressure_gas_similarity','filling_gas_similarity','pressure_residual','cushion_residual'],
'normal_pressure_filling_residual4':['pressure_normal_distance','filling_normal_distance','pressure_residual','cushion_residual'],
'std_cushion_barrel5_pressure_cushion_residual4':[oldcols[2],oldcols[3],'pressure_residual','cushion_residual'],
'std_barrel2_hopper_plasticizing_cushion_residual4':[oldcols[0],oldcols[1],'plasticizing_residual','cushion_residual'],
'std_cushion_barrel5_thermal_distance_mold_residual4':[oldcols[2],oldcols[3],'thermal_normal_distance','mold_residual'],
'previous_std10':oldcols,
'raw4':rawcols}
def learned_features(ref_idx,query_idx):
    # Every reference index is outside query block, both outer and inner CV.
    assert set(groups.iloc[ref_idx]).isdisjoint(groups.iloc[query_idx])
    ref=data.iloc[ref_idx];q=data.iloc[query_idx]
    normal=ref.loc[y.iloc[ref_idx].eq(0).to_numpy()]
    gas=ref.loc[y.iloc[ref_idx].eq(1).to_numpy()]
    out=pd.DataFrame(index=query_idx)
    for group,cols in sensor_groups.items():
        sc=RobustScaler().fit(normal[cols])
        xn=sc.transform(normal[cols]);xq=sc.transform(q[cols])
        cov=LedoitWolf().fit(xn)
        out[group+'_normal_distance']=np.sqrt(np.maximum(cov.mahalanobis(xq),0))
        dn=NearestNeighbors(n_neighbors=min(3,len(normal))).fit(xn).kneighbors(xq)[0].mean(axis=1)
        if len(gas):
            xg=sc.transform(gas[cols])
            dg=NearestNeighbors(n_neighbors=min(3,len(gas))).fit(xg).kneighbors(xq)[0].mean(axis=1)
            out[group+'_gas_similarity']=np.log((dn+1e-6)/(dg+1e-6))
        else:out[group+'_gas_similarity']=np.nan
    for name,(target,predictors) in relations.items():
        sc=RobustScaler().fit(normal[predictors])
        model=Ridge(alpha=10).fit(sc.transform(normal[predictors]),normal[target])
        fitted=model.predict(sc.transform(normal[predictors]))
        scale=np.sqrt(np.mean((normal[target].to_numpy()-fitted)**2))
        floor=max(float(normal[target].std(ddof=0))*.1,1e-6)
        out[name]=(q[target].to_numpy()-model.predict(sc.transform(q[predictors])))/max(scale,floor)
    return out
# Outer heldout blocks never contribute to feature fitting.
# Training learned features are inner leave-block-out to avoid same-shot/self neighbor shortcuts.
fold_data=[];audit=[]
outer=list(LeaveOneGroupOut().split(data,y,groups))
for train,test in outer:
    train_features=static.iloc[train].copy()
    test_features=static.iloc[test].copy()
    for block in groups.iloc[train].unique():
        query=train[groups.iloc[train].eq(block).to_numpy()]
        ref=train[~groups.iloc[train].eq(block).to_numpy()]
        feat=learned_features(ref,query)
        for c in feat:train_features.loc[query,c]=feat[c]
        audit.append(dict(outer_test=str(groups.iloc[test[0]]),inner_query=str(block),reference_shots=len(ref),reference_gas=int(y.iloc[ref].sum())))
    testlearn=learned_features(train,test)
    for c in testlearn:test_features[c]=testlearn[c]
    fold_data.append((train,test,train_features,test_features))
    print('feature fold prepared',str(groups.iloc[test[0]]),flush=True)
results=[];prediction_parts=[];blockrows=[]
for setting,cols in configs.items():
    probs=np.full(len(data),np.nan)
    for train,test,tr,te in fold_data:
        imp=SimpleImputer(strategy='median',keep_empty_features=True)
        xt=imp.fit_transform(tr[cols]);xv=imp.transform(te[cols])
        model=RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=-1)
        model.fit(xt,y.iloc[train]);probs[test]=model.predict_proba(xv)[:,list(model.classes_).index(1)]
    assert np.isfinite(probs).all()
    predicted=probs>=.5;tn,fp,fn,tp=confusion_matrix(y,predicted,labels=[0,1]).ravel()
    selected=np.zeros(len(data),dtype=bool)
    tie=np.random.RandomState(42).random_sample(len(data))
    for block in groups.unique():
        idx=np.flatnonzero(groups.eq(block).to_numpy())
        order=idx[np.lexsort((tie[idx],-probs[idx]))]
        selected[order[:int(np.ceil(len(idx)*.1))]]=True
        blockrows.append(dict(setting=setting,block=block,n=len(idx),gas=int(y.iloc[idx].sum()),TP=int((predicted[idx]&y.iloc[idx].astype(bool)).sum()),FP=int((predicted[idx]&~y.iloc[idx].astype(bool)).sum()),caught_top10=int((selected[idx]&y.iloc[idx].astype(bool)).sum())))
    row=dict(setting=setting,features=len(cols),shots=len(data),gas=int(y.sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),AP=average_precision_score(y,probs),F1=f1_score(y,predicted,zero_division=0),precision=precision_score(y,predicted,zero_division=0),recall=recall_score(y,predicted,zero_division=0),caught_top10=int((selected&y.astype(bool)).sum()),selected_top10=int(selected.sum()))
    results.append(row)
    pred=data[['EQUIP_CD','product','TimeStamp','comparison_block','Reason']].copy()
    pred['gas']=y;pred['probability']=probs;pred['predicted_gas']=predicted;pred['selected_top10']=selected;pred['setting']=setting
    prediction_parts.append(pred)
    pd.DataFrame(results).to_csv(tables/'rg3_relational_feature_metrics.csv',index=False,encoding='utf-8-sig')
    print(setting,row,flush=True)
metrics=pd.DataFrame(results);predictions=pd.concat(prediction_parts,ignore_index=True)
assert len(metrics)==14 and len(predictions)==8162
assert not predictions.duplicated(['setting','EQUIP_CD','product','TimeStamp']).any()
assert (metrics.TP+metrics.FN==17).all() and (metrics[['TP','FP','FN','TN']].sum(axis=1)==583).all()
baseline=metrics.loc[metrics.setting.eq('previous_std10')].iloc[0]
assert baseline.TP==1 and baseline.FP==6 and abs(baseline.AP-.13871113959502832)<1e-10
predictions.to_csv(tables/'rg3_relational_feature_oof_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(blockrows).to_csv(tables/'rg3_relational_feature_by_block.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(audit).to_csv(tables/'rg3_relational_feature_nested_audit.csv',index=False,encoding='utf-8-sig')
pd.DataFrame([dict(setting=k,features=len(v),columns=';'.join(v)) for k,v in configs.items()]).to_csv(tables/'rg3_relational_feature_inputs.csv',index=False,encoding='utf-8-sig')
# For diagnostics, heldout feature values only; never re-used as training inputs across folds.
held=pd.concat([te.assign(outer_test=str(groups.iloc[test[0]])) for train,test,tr,te in fold_data]).sort_index()
held=held.join(data[['TimeStamp','comparison_block','Reason']]).assign(gas=y)
held.to_csv(tables/'rg3_relational_features_heldout.csv',index=False,encoding='utf-8-sig')
comparison=[]
for block,part in held.groupby('comparison_block'):
    for col in shape+phase+distcols+nearcols+rescols:
        n=part.loc[part.gas.eq(0),col].dropna();bad=part.loc[part.gas.eq(1),col].dropna()
        q10,q90=n.quantile(.1),n.quantile(.9)
        comparison.append(dict(block=block,feature=col,normal_n=len(n),gas_n=len(bad),normal_median=n.median(),gas_median=bad.median(),difference=bad.median()-n.median(),gas_outside_middle80=int(((bad<q10)|(bad>q90)).sum())))
pd.DataFrame(comparison).to_csv(tables/'rg3_relational_feature_overlap.csv',index=False,encoding='utf-8-sig')
names={
'thermal_shape4':'배럴 온도구배3개+금형온도차',
'time_phase4':'사출−충전시간,사이클−가소화시간,가소화비율,충전비율',
'normal_relation_residual4':'정상관계 예측잔차4개',
'normal_multivariate_distance4':'공정별 정상 다변량거리4개',
'gas_similarity4':'공정별 가스대비정상 상대거리4개',
'pressure_gas_thermal_shape3':'압력가스유사도+배럴온도구배2개',
'filling_plasticizing_similarity2':'충전·가소화 가스유사도2개',
'pressure_filling_similarity_residual4':'압력·충전 가스유사도+사출압력·쿠션잔차',
'normal_pressure_filling_residual4':'압력·충전 정상거리+사출압력·쿠션잔차',
'std_cushion_barrel5_pressure_cushion_residual4':'쿠션·배럴5흔들림+압력·쿠션잔차',
'std_barrel2_hopper_plasticizing_cushion_residual4':'배럴2·호퍼흔들림+가소화·쿠션잔차',
'std_cushion_barrel5_thermal_distance_mold_residual4':'쿠션·배럴5흔들림+온도정상거리·금형잔차',
'previous_std10':'기존10개 온도·쿠션흔들림',
'raw4':'평균배압·최대사출압력·쿠션·배럴5 원값'}
lines=['# RG3 가스: 센서 관계와 정상·가스 상대거리','',
'정상566 Shot,가스17 Shot. 5관측구간 전체제외 평가. 동일RF(200트리,깊이8,말단최소3,balanced_subsample,seed42),확률0.5 고정.',
'','## 새로운 가설','',
'- 온도가 전체적으로 높거나낮은것보다 배럴온도배치의형태가다를수있다. 배럴1−3,4−5,5−6 및 금형4−3 차이.',
'- 단일센서값이정상이어도 함께움직이는센서관계가깨질수있다. 학습정상만으로Ridge 관계를맞추고 현재값의잔차를정상잔차폭으로나눈다.',
'- 정상분포에서멀다고가스는아니다. 압력·온도·충전·가소화각그룹에서 정상과가스까지의거리를따로계산하여 상대적유사도를사용한다. GitHub의불량근접거리아이디어를파생점수로구체화했으며 비라벨에가짜정답을붙이지않았다.',
'','## 학습형 파생변수의 누수 방지','',
'평가구간은 스케일러·공분산·회귀·최근접가스선정에 절대사용하지않는다. RF학습입력도 학습구간내에서 관측구간을하나씩빼는 추가분할로 생성하여 자기자신 또는 같은구간의가스를참고하지않는다. 평가입력은 나머지4구간으로생성한다. label-dependent feature는 단순센서파생값이아니라 학습형점수/2단계모델이다.',
'',
'공정그룹별4개입력은 원센서14개를요약한것이므로 단순히 센서4개만사용하는모델은아니다. learned residual은다른원센서도사용한다. 최대/평균배압·RPM정의불일치가있어 차이를변동폭으로사용하지않았다.',
'','## 계산','',
'정상거리: 각그룹의학습정상으로RobustScaler와LedoitWolf공분산을학습한Mahalanobis거리.',
'',
'가스유사도: log((가까운정상3개평균거리+1e-6)/(가까운가스최대3개평균거리+1e-6)). 양수가크면해당센서그룹에서정상보다가스에상대적으로가깝다는뜻이다. 스케일은학습정상으로만학습한다.',
'',
'잔차4개: 최대사출압력~사출속도+충전시간,가소화시간~평균RPM+평균배압,쿠션~가소화위치+최대사출압력,금형4~금형3+배럴5. Ridge alpha10고정. 물리적법칙으로확정한식이아닌통계관계가설이다.',
'',
'시간차·비율은현재Shot에서계산한다. 사이클−가소화시간을냉각시간으로해석하지않으며 시간컬럼정의는추가확인이필요하다.',
'','## 결과','',
'| 조합 | 입력수 | 불량판정 | TP | FP | 정상판정 | TN | FN | AP | F1 | 상위62개검사 발견/17 |',
'|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for r in metrics.itertuples():lines.append(f'| {names[r.setting]} | {r.features} | {r.TP+r.FP} | {r.TP} | {r.FP} | {r.TN+r.FN} | {r.TN} | {r.FN} | {r.AP:.4f} | {r.F1:.4f} | {r.caught_top10} |')
lines+=['','## 모든 가스 Shot의 평가 예측','',
'새후보중AP최고설정은 동일평가자료로탐색해고른것이며 독립최종성능이아니다.',
'','| 시각 | 실제 | 탐색최고설정 | 가스확률 | 0.5판정 | 상위10%검사 |','|---|---|---|---:|---|---|']
new=metrics.loc[~metrics.setting.isin(['previous_std10','raw4'])]
best=new.sort_values('AP',ascending=False).iloc[0]
for r in predictions.loc[predictions.setting.eq(best.setting)&predictions.gas.eq(1)].itertuples():lines.append(f'| {r.TimeStamp} | 가스 | {best.setting} | {r.probability:.4f} | {"가스" if r.predicted_gas else "정상"} | {"포함" if r.selected_top10 else "제외"} |')
lines+=['','## 해석 한계','',
'파생변수탐색에동일평가결과를사용했다. 17개가스는4개관측구간에몰려있고 관측구간은 실제Run ID가아니다. 학습에미래구간이포함될수있어 실제미래성능시험과다르다. nested score계산은누수를줄이지만외부독립검증을대체하지않는다. 정상관계이탈이나가스유사도는가스발생원인의증거가아니다.']
(root/'reports/rg3_relational_feature_comparison.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('VERIFIED 14 models; nested feature separation; baseline reproduced.',flush=True)



feature fold prepared 20201021_001626_RG3


feature fold prepared 20201022_002405_RG3


feature fold prepared 20201022_044026_RG3


feature fold prepared 20201023_010128_RG3


feature fold prepared 20201023_044148_RG3


...

std_barrel2_hopper_plasticizing_cushion_residual4 {'setting': 'std_barrel2_hopper_plasticizing_cushion_residual4', 'features': 4, 'shots': 583, 'gas': 17, 'TP': 0, 'FP': 1, 'FN': 17, 'TN': 565, 'AP': 0.03829119559678267, 'F1': 0.0, 'precision': 0.0, 'recall': 0.0, 'caught_top10': 2, 'selected_top10': 62}


std_cushion_barrel5_thermal_distance_mold_residual4 {'setting': 'std_cushion_barrel5_thermal_distance_mold_residual4', 'features': 4, 'shots': 583, 'gas': 17, 'TP': 0, 'FP': 0, 'FN': 17, 'TN': 566, 'AP': 0.03595757176040192, 'F1': 0.0, 'precision': 0.0, 'recall': 0.0, 'caught_top10': 3, 'selected_top10': 62}


previous_std10 {'setting': 'previous_std10', 'features': 4, 'shots': 583, 'gas': 17, 'TP': 1, 'FP': 6, 'FN': 16, 'TN': 560, 'AP': 0.13871113959502832, 'F1': 0.08333333333333333, 'precision': 0.14285714285714285, 'recall': 0.058823529411764705, 'caught_top10': 6, 'selected_top10': 62}


raw4 {'setting': 'raw4', 'features': 4, 'shots': 583, 'gas': 17, 'TP': 1, 'FP': 12, 'FN': 16, 'TN': 554, 'AP': 0.05131591669752247, 'F1': 0.06666666666666667, 'precision': 0.07692307692307693, 'recall': 0.058823529411764705, 'caught_top10': 3, 'selected_top10': 62}


VERIFIED 14 models; nested feature separation; baseline reproduced.
